## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv

## loading all the environment variables
load_dotenv() 

groq_api_key = os.getenv("GROQ_API_KEY")
print(groq_api_key)

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


In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "llama-3.1-8b-instant", groq_api_key = groq_api_key)
model

d:\Data Science and Gen AI Bootcamp\Udemy Material\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E7F3BB4700>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E7F3BB4DF0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
## invoking the model with a list of messages using HumanMessage
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content = "Hi , My name is Dhruv and I am a student at KU Leuven, Belgium")])

AIMessage(content='Nice to meet you, Dhruv. KU Leuven is a prestigious university in Belgium, known for its strong academic programs and research opportunities. Which faculty or department are you studying in, if I might ask?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 56, 'total_tokens': 102, 'completion_time': 0.060426072, 'completion_tokens_details': None, 'prompt_time': 0.003725038, 'prompt_tokens_details': None, 'queue_time': 0.018839777, 'total_time': 0.06415111}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d77f8-bf9f-7c02-ae06-e4af9f4bd0fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 46, 'total_tokens': 102})

In [4]:
## AIMessage is used to represent the response from the model
## we can also use it to provide context to the model about the previous conversation. This is useful for maintaining the context of the conversation and making the responses more relevant.
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content = "Hi , My name is Dhruv and I am a student at KU Leuven, Belgium"),
        AIMessage(content = "Nice to meet you, Dhruv. KU Leuven is a prestigious university in Belgium, known for its strong academic programs and research opportunities. Which faculty or department are you studying in, if I might ask?"),
        HumanMessage(content = "Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Dhruv, and if I recall correctly, you're a student at KU Leuven in Belgium.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 122, 'total_tokens': 149, 'completion_time': 0.035094261, 'completion_tokens_details': None, 'prompt_time': 0.006625357, 'prompt_tokens_details': None, 'queue_time': 0.017627974, 'total_time': 0.041719618}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8639719ff2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d77fa-f380-7791-b5fe-913086e3bbe5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 27, 'total_tokens': 149})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
## ChatMessageHistory is a class that can be used to store the history of the conversation.
## It can be used to maintain the context of the conversation and make the responses more relevant.
from langchain_community.chat_message_histories import ChatMessageHistory

## BaseChatMessageHistory is an abstract class that defines the interface for the chat message history.
## It can be used to create custom implementations of the chat message history.
from langchain_core.chat_history import BaseChatMessageHistory

## RunnableWithMessageHistory is a class that can be used to create a runnable that maintains the message history.
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

## get_session_history is a function that can be used to get the chat message history for a given session id.
## If the session id does not exist in the store, it creates a new chat message history and stores it in the store.
def get_session_history(session_id : str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

## Interact with the model using the chat message history.
## The model will maintain the context of the conversation and make the responses more relevant.
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [9]:
## Now we can invoke the model using the with_message_history runnable.
## We need to pass the session_id in the config to maintain the context of the conversation.
config = {
    "configurable" : {
        "session_id" : "chat1"
    }
}

response = with_message_history.invoke(
    [HumanMessage(content = "Hi, My name is Dhruv and I am a student at KU Leuven, Belgium")],
    config = config
)

print('Response:', response)
print('Content:', response.content)

Response: content="Hello Dhruv. It seems like you introduced yourself, but didn't specify what you're studying or what you'd like to talk about. If you're open to conversation, I can suggest some topics. Would you like to talk about university life in Leuven, your studies, or something else?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 221, 'total_tokens': 284, 'completion_time': 0.107284901, 'completion_tokens_details': None, 'prompt_time': 0.012095546, 'prompt_tokens_details': None, 'queue_time': 0.018641612, 'total_time': 0.119380447}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8639719ff2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d7804-b53f-77d3-8089-d6171e785323-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 221, 'output_tokens': 63, 'total_tokens': 284}
Content: Hello Dhruv. It seems like you introduced yourself

In [10]:
with_message_history.invoke(
    [HumanMessage(content = "What's my name?")],
    config = config,
)

AIMessage(content='Your name is Dhruv.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 298, 'total_tokens': 306, 'completion_time': 0.004173962, 'completion_tokens_details': None, 'prompt_time': 0.018763159, 'prompt_tokens_details': None, 'queue_time': 0.0450176, 'total_time': 0.022937121}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e09ee421cf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d7805-7902-70f2-9500-e5f8987e9b4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'output_tokens': 8, 'total_tokens': 306})

In [11]:
## change the config --> session id
config1 = {
    "configurable": {
        "session_id" : "chat2"
    }
}

## Change the session id to chat2, so the model will not have the context of the previous conversation
## It will not be able to answer the question correctly.
response = with_message_history.invoke(
    [HumanMessage(content = "What's my name?")],
    config = config1
)

print(response.content)

I don't have any information about your name. I'm a large language model, I don't retain any personal data or information about individual users. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!


In [13]:
response = with_message_history.invoke(
    [HumanMessage(content = "Hey My name is XYZ")],
    config = config1
)
print('Message before chat history:', response.content)

## Now the model has the context of the previous conversation
## so it will be able to answer the question correctly.
response = with_message_history.invoke(
    [HumanMessage(content= "What's my name?")],
    config = config1
)

print('Message after chat history:', response.content)

Message before chat history: We've had this conversation before. Your name is still XYZ.
Message after chat history: Still XYZ. I don't have any updates to your name.


### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

## MessagesPlaceholder is a class that can be used to represent the messages in the prompt template.
## It allows us to pass the messages as a variable in the prompt template.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name = "messages")
    ]
)

chain = prompt | model

In [15]:
## Key value pair to pass the messages to the prompt template, with messages as the key and a list of HumanMessage as the value.
chain.invoke({"messages" : [HumanMessage(content = "Hi My name is Dhruv!")]})

AIMessage(content="Nice to meet you, Dhruv! I'm happy to be your assistant and help you with any questions or tasks you may have. What's on your mind today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 59, 'total_tokens': 95, 'completion_time': 0.058888771, 'completion_tokens_details': None, 'prompt_time': 0.003155677, 'prompt_tokens_details': None, 'queue_time': 0.044986525, 'total_time': 0.062044448}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d9492c3c54', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d780c-83f5-74c2-9c93-68ef51291201-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 36, 'total_tokens': 95})

In [ ]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

config = {
    "configurable": {
        "session_id": "chat3"
    }
}

response1 = with_message_history.invoke(
    [HumanMessage(content = "Hi My name is Dhruv!")],
    config = config
)

print(response1)

## Now the model has the context of the previous conversation 
## so it will be able to answer the question correctly.
response2 = with_message_history.invoke(
    [HumanMessage(content = "What's my name?")],
    config = config,
)

print(response2.content)

content='Small world, Dhruv! It seems we already met a moment ago. Is there something I can help you with, or would you like to chat about something in particular?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 111, 'total_tokens': 148, 'completion_time': 0.071617438, 'completion_tokens_details': None, 'prompt_time': 0.006401358, 'prompt_tokens_details': None, 'queue_time': 0.018134351, 'total_time': 0.078018796}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d780f-62b0-7913-aeb1-ac760cab02e5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 111, 'output_tokens': 37, 'total_tokens': 148}
Your name is Dhruv.


In [18]:
## Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all questions to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name = "messages"),
    ]
)

chain = prompt | model

In [ ]:
response = chain.invoke({
        "messages" : [HumanMessage(content = "Hi My name is Dhruv!")], "language" : "French"
})

## The model will answer the question in French as we have specified the language in the prompt template.
print(response.content)

Bonjour Dhruv ! Enchanté de vous rencontrer ! Comment allez-vous aujourd'hui ?


Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key = "messages"
)

In [ ]:
config = {
    "configurable": {
        "session_id": "chat4"
    }
}

response = with_message_history.invoke(
    {'messages': [HumanMessage(content = "Hi,I am Dhruv!")], "language" : "French"},
    config = config
)

print(response.content)

Bonjour Dhruv, comment allez-vous aujourd'hui ? Je suis ravi de vous aider avec tout ce que vous pourriez me demander. Qu'est-ce que vous désirez savoir ou faire aujourd'hui ?


In [24]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content = "whats my name?")], "language": "Hindi"},
    config = config,
)

response.content

'Dhruv, तुम्हारा नाम ध्रुव है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [25]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens = 45,
    strategy = "last",
    token_counter = model,
    include_system = True,
    allow_partial = False,
    start_on = "human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

d:\Data Science and Gen AI Bootcamp\Udemy Material\Langchain\venv\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
d:\Data Science and Gen AI Bootcamp\Udemy Material\Langchain\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhruv\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you ei

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [26]:
from operator import itemgetter

## RunnablePassthrough is a class that can be used to create a runnable that does not do anything to the input
## Just passes it through to the next runnable in the chain.
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
    "messages" : messages + [HumanMessage(content = "What ice cream do i like")],
    "language":"English"
    }
)
response.content

"I don't have that information. We just started chatting, I don't have any prior knowledge about your preferences. Would you like to tell me what your favorite ice cream flavor is?"

In [27]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content = "what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked the math problem "2 + 2".'

In [28]:
## Lets wrap this in the Message History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable" : {
    "session_id" : "chat5"
}}

In [43]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.  \n\nIf you'd like to tell me your name, I'd be happy to know! 😊  \n\n"

In [29]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You haven't asked a math problem yet. This conversation just started, so I'm ready to help with any math or other questions you might have. What's on your mind?"